# Notebook 2: Drug–Food Interaction Model (Kaggle Dataset)

This notebook trains a **food-behavior prediction model** using the Kaggle drug–food interaction dataset.

### Pipeline:
1. Load the Kaggle JSON dataset (1,423 drugs with food interaction text)
2. For each drug, look up its SMILES string in the PyTDC DDI DrugBank dataset
3. Use a **BERT-based NLP model** (PubMedBERT / BioBERT) to extract structured food-behavior labels from the free-text interaction strings
4. Encode SMILES as Morgan fingerprints → train a **multi-label classifier**
5. Evaluate on train / val / test splits
6. Save model + lookup table to Google Drive

In [ ]:
# ─── Install Dependencies ───────────────────────────────────────────────────
!pip install -q 'numpy<2' rdkit scikit-learn PyTDC transformers torch kagglehub matplotlib seaborn

In [ ]:
# ─── Imports ────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json, os, re, pickle
from collections import defaultdict, Counter

import torch
from transformers import AutoTokenizer, AutoModel

from rdkit import Chem
from rdkit.Chem import AllChem

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report
)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

In [ ]:
# ─── Mount Google Drive ──────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = '/content/drive/MyDrive/drug_interaction_models'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Models will be saved to: {SAVE_DIR}')

---
## Step 1 — Load Kaggle Drug–Food Dataset

In [ ]:
# ─── 1.1  Download Kaggle Dataset ────────────────────────────────────────────
import kagglehub

food_path = kagglehub.dataset_download('shayanhusain/drug-food-interactions-dataset')
print('Downloaded to:', food_path)

# Find the JSON file
for root, dirs, files in os.walk(food_path):
    for f in files:
        print(os.path.join(root, f))

In [ ]:
# ─── 1.2  Load JSON ──────────────────────────────────────────────────────────
import glob
json_files = glob.glob(os.path.join(food_path, '**', '*.json'), recursive=True)
json_path = json_files[0]
print('Loading:', json_path)

with open(json_path, 'r') as f:
    food_raw = json.load(f)

print(f'Total records: {len(food_raw)}')
print('Sample record:', food_raw[0])

In [ ]:
# ─── 1.3  Flatten to DataFrame ───────────────────────────────────────────────
food_df = pd.DataFrame(food_raw)
# Explode: each row = one drug + its list of food_interaction strings
food_df['food_interactions'] = food_df['food_interactions'].apply(
    lambda x: x if isinstance(x, list) else [str(x)]
)
print(f'Drugs with food interaction data: {len(food_df)}')
print(food_df[['name', 'food_interactions']].head())

---
## Step 2 — BERT-based Label Extraction from Free Text

We define a rule-based + BERT embedding approach to map free-text food interaction strings
into structured behavior categories that a downstream classifier can predict from SMILES.

In [ ]:
# ─── 2.1  Define Food Behavior Categories ────────────────────────────────────
# These categories cover the major food-drug interaction archetypes
FOOD_CATEGORIES = [
    'avoid_alcohol',
    'avoid_grapefruit',
    'take_with_food',
    'take_on_empty_stomach',
    'avoid_dairy',
    'avoid_high_fat_meals',
    'avoid_vitamin_k_foods',   # warfarin / anticoagulants
    'avoid_tyramine_foods',    # MAOIs
    'avoid_herbal_supplements',
    'limit_caffeine',
    'increase_fluid_intake',
    'avoid_salt_substitutes',
    'general_dietary_warning',
]

# Keyword patterns for each category (regex)
CATEGORY_PATTERNS = {
    'avoid_alcohol':           r'alcohol|ethanol|drink|beer|wine|spirits',
    'avoid_grapefruit':        r'grapefruit|pomelo',
    'take_with_food':          r'take with food|with meals|with a meal|with or after food|eat before|with water and food',
    'take_on_empty_stomach':   r'empty stomach|before meals|fasting|without food|1 hour before|30 minutes before eating',
    'avoid_dairy':             r'dairy|milk|cheese|calcium|antacid',
    'avoid_high_fat_meals':    r'high.fat|fatty meal|high fat',
    'avoid_vitamin_k_foods':   r'vitamin k|leafy green|spinach|kale|broccoli|cabbage|anticoagulant',
    'avoid_tyramine_foods':    r'tyramine|aged cheese|fermented|cured meat|sauerkraut|soy sauce',
    'avoid_herbal_supplements': r'herb|supplement|ginkgo|ginseng|garlic|ginger|chamomile|st. john|echinacea|valerian|supplement',
    'limit_caffeine':          r'caffeine|coffee|tea|cola|energy drink',
    'increase_fluid_intake':   r'fluid|water|hydrat|drink plenty',
    'avoid_salt_substitutes':  r'salt substitute|potassium|sodium',
    'general_dietary_warning': r'diet|food|nutritional|eating|meal|consume',
}

def classify_food_text(text_list):
    """Return a list of category labels for a drug's food_interactions strings."""
    combined = ' '.join(text_list).lower()
    matched  = []
    for cat, pattern in CATEGORY_PATTERNS.items():
        if re.search(pattern, combined):
            matched.append(cat)
    # Fallback: if nothing matched, assign general warning
    if not matched and combined.strip():
        matched = ['general_dietary_warning']
    return matched

# Apply to all drugs
food_df['behavior_labels'] = food_df['food_interactions'].apply(classify_food_text)

# Show label distribution
all_labels = [lbl for lbls in food_df['behavior_labels'] for lbl in lbls]
label_counts = Counter(all_labels)

fig, ax = plt.subplots(figsize=(12, 5))
cats = list(label_counts.keys())
cnts = list(label_counts.values())
ax.barh(cats[::-1], cnts[::-1], color='steelblue')
ax.set_xlabel('Number of Drugs')
ax.set_title('Food-Drug Dataset — Behavior Category Distribution')
plt.tight_layout()
plt.savefig('/content/food_label_distribution.png', dpi=120)
plt.show()

print('\nLabel counts:', dict(label_counts))

---
## Step 3 — SMILES Lookup via PyTDC DDI DrugBank

In [ ]:
# ─── 3.1  Look Up SMILES via PubChemPy ──────────────────────────────────────
# The Kaggle dataset has common drug names (e.g. "Lepirudin", "Aspirin").
# DrugBank IDs (DB04571 etc.) are internal codes that do NOT match names,
# so we go straight to PubChem which accepts common/brand/generic names.
!pip install -q pubchempy
import pubchempy as pcp
import time

def pubchem_smiles(name):
    """Return isomeric SMILES for a drug name via PubChemPy, or None."""
    try:
        compounds = pcp.get_compounds(name, "name", timeout=10)
        if compounds:
            smi = compounds[0].isomeric_smiles
            return smi if smi else compounds[0].canonical_smiles
    except Exception:
        pass
    return None

drug_smiles_map = {}   # drug_name -> SMILES
missing = []

print(f"Looking up SMILES for {len(food_df)} drugs via PubChem...")
print("This calls the PubChem API once per drug; expect ~15-25 min for 1,423 drugs.")

for i, row in enumerate(food_df.itertuples(), 1):
    name  = row.name
    smiles = pubchem_smiles(name)
    if smiles:
        drug_smiles_map[name] = smiles
    else:
        missing.append(name)
    # Polite rate limiting (PubChem allows ~5 req/sec without API key)
    time.sleep(0.25)
    if i % 100 == 0 or i == len(food_df):
        print(f"  {i}/{len(food_df)} done — found: {len(drug_smiles_map)}, missing: {len(missing)}")

print("Final — Found SMILES: {len(drug_smiles_map)} | No SMILES: {len(missing)}")
print("Missing (first 10):", missing[:10])

In [ ]:
# ─── 3.2  Filter to Drugs with SMILES & Save Lookup Table ───────────────────
food_df["smiles"] = food_df["name"].map(drug_smiles_map)
food_df_valid = food_df.dropna(subset=["smiles"]).copy()

print(f"Records with SMILES (usable for training): {len(food_df_valid)}")
print(f"Records without SMILES (lookup-table only): {len(food_df) - len(food_df_valid)}")

# Save full lookup table (includes drugs without SMILES so lookup mode still works)
food_lookup = {
    row["name"]: {
        "smiles":            row.get("smiles"),
        "food_interactions": row["food_interactions"],
        "behavior_labels":   row["behavior_labels"]
    }
    for _, row in food_df.iterrows()
}
with open("/content/food_lookup.json", "w") as f:
    json.dump(food_lookup, f)
print("Saved food_lookup.json")

---
## Step 4 — BERT Embedding of Food Interaction Text

We use **BioBERT** (`dmis-lab/biobert-base-cased-v1.2`) to encode each drug's
food interaction text into a dense embedding.  These embeddings are used as
additional features alongside the Morgan fingerprints.

In [ ]:
# ─── 4.1  Load BioBERT ───────────────────────────────────────────────────────
BERT_MODEL = 'dmis-lab/biobert-base-cased-v1.2'
print(f'Loading {BERT_MODEL}...')

bert_tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL)
bert_model     = AutoModel.from_pretrained(BERT_MODEL).to(DEVICE)
bert_model.eval()
print('BioBERT loaded.')

In [ ]:
# ─── 4.2  Encode Food Text to Embeddings ─────────────────────────────────────
def bert_encode(texts, batch_size=32, max_length=128):
    """Return mean-pooled [CLS] embeddings for a list of strings."""
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc   = bert_tokenizer(
            batch, padding=True, truncation=True,
            max_length=max_length, return_tensors='pt'
        ).to(DEVICE)
        with torch.no_grad():
            out = bert_model(**enc)
        # Mean pool over non-padding tokens
        mask = enc['attention_mask'].unsqueeze(-1).float()
        emb  = (out.last_hidden_state * mask).sum(1) / mask.sum(1)
        all_embeddings.append(emb.cpu().numpy())
    return np.vstack(all_embeddings)

# Concatenate all food interaction strings per drug for encoding
food_texts = food_df_valid['food_interactions'].apply(lambda x: ' | '.join(x)).tolist()
print(f'Encoding {len(food_texts)} drug food-interaction texts with BioBERT...')
food_bert_embeddings = bert_encode(food_texts)
print(f'BERT embedding shape: {food_bert_embeddings.shape}')

---
## Step 5 — Feature Engineering & Train/Val/Test Split

In [ ]:
# ─── 5.1  Morgan Fingerprints for Kaggle Drugs ───────────────────────────────
def smiles_to_fp(smiles, radius=2, nbits=2048):
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        return np.zeros(nbits)
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=nbits)
    return np.array(fp)

food_fps = np.stack(food_df_valid['smiles'].apply(smiles_to_fp).values)
print(f'Fingerprint matrix shape: {food_fps.shape}')

# Combined feature: fingerprint + BERT embedding
X_food = np.hstack([food_fps, food_bert_embeddings])
print(f'Combined feature matrix shape: {X_food.shape}')

In [ ]:
# ─── 5.2  Binarise Multi-labels ──────────────────────────────────────────────
food_mlb = MultiLabelBinarizer(classes=FOOD_CATEGORIES)
y_food   = food_mlb.fit_transform(food_df_valid['behavior_labels'])

print(f'Number of food behavior classes: {y_food.shape[1]}')
print(f'Label matrix shape: {y_food.shape}')

In [ ]:
# ─── 5.3  Split into Train / Val / Test ──────────────────────────────────────
# 70% train, 15% val, 15% test
X_tr, X_tmp, y_tr, y_tmp = train_test_split(
    X_food, y_food, test_size=0.30, random_state=42
)
X_va, X_te, y_va, y_te = train_test_split(
    X_tmp, y_tmp, test_size=0.50, random_state=42
)

print(f'Train: {X_tr.shape[0]} | Val: {X_va.shape[0]} | Test: {X_te.shape[0]}')

---
## Step 6 — Train Food-Behavior Model

In [ ]:
# ─── 6.1  Train ──────────────────────────────────────────────────────────────
# Note: this dataset is small (~1000 drugs), so we use a lighter model
food_base_clf = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    min_samples_leaf=2,
    class_weight='balanced',
    n_jobs=-1,
    random_state=42
)
food_model = OneVsRestClassifier(food_base_clf, n_jobs=-1)

print('Training food-behavior multi-label model...')
food_model.fit(X_tr, y_tr)
print('Training complete.')

In [ ]:
# ─── 6.2  Evaluation Helper ──────────────────────────────────────────────────
def evaluate_food_model(model, X, y_true, split_name=''):
    y_pred   = model.predict(X)
    micro_f1 = f1_score(y_true, y_pred, average='micro', zero_division=0)
    macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    sample_f1 = f1_score(y_true, y_pred, average='samples', zero_division=0)
    exact_acc = accuracy_score(y_true, y_pred)
    print(f'\n=== {split_name} ===')
    print(f'  Micro F1     : {micro_f1:.4f}')
    print(f'  Macro F1     : {macro_f1:.4f}')
    print(f'  Sample F1    : {sample_f1:.4f}')
    print(f'  Exact Match  : {exact_acc:.4f}')
    return {
        'split': split_name, 'micro_f1': micro_f1, 'macro_f1': macro_f1,
        'sample_f1': sample_f1, 'exact_match': exact_acc, 'y_pred': y_pred
    }

food_results = {}
food_results['train'] = evaluate_food_model(food_model, X_tr, y_tr, 'Train')
food_results['val']   = evaluate_food_model(food_model, X_va, y_va, 'Validation')
food_results['test']  = evaluate_food_model(food_model, X_te, y_te, 'Test')

In [ ]:
# ─── 6.3  Metrics Bar Chart ───────────────────────────────────────────────────
food_metrics = pd.DataFrame([
    {k: v for k, v in r.items() if k in ['split', 'micro_f1', 'macro_f1', 'sample_f1', 'exact_match']}
    for r in food_results.values()
])

fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(food_metrics))
w = 0.2
ax.bar(x - 1.5*w, food_metrics['micro_f1'],  w, label='Micro F1',    color='steelblue')
ax.bar(x - 0.5*w, food_metrics['macro_f1'],  w, label='Macro F1',    color='seagreen')
ax.bar(x + 0.5*w, food_metrics['sample_f1'], w, label='Sample F1',   color='coral')
ax.bar(x + 1.5*w, food_metrics['exact_match'],w, label='Exact Match', color='orchid')
ax.set_xticks(x)
ax.set_xticklabels(food_metrics['split'])
ax.set_ylim(0, 1.05)
ax.set_ylabel('Score')
ax.set_title('Food-Behavior Model — Performance by Split (⚠️ Small dataset — see notes)')
ax.legend()
plt.tight_layout()
plt.savefig('/content/food_metrics_bar.png', dpi=120)
plt.show()

print('NOTE: This dataset is small (~1000 usable drugs). Model predictions may be unreliable.')
print('The inference notebook includes a lookup-table fallback for this reason.')

In [ ]:
# ─── 6.4  Per-class F1 (Test Set) ────────────────────────────────────────────
per_class_f1 = f1_score(y_te, food_results['test']['y_pred'], average=None, zero_division=0)

fig, ax = plt.subplots(figsize=(12, 5))
ax.barh(FOOD_CATEGORIES[::-1], per_class_f1[::-1], color='teal')
ax.set_xlabel('F1 Score')
ax.set_title('Food-Behavior Model — Per-class F1 Scores (Test Set)')
ax.set_xlim(0, 1)
plt.tight_layout()
plt.savefig('/content/food_per_class_f1.png', dpi=120)
plt.show()

In [ ]:
# ─── 6.5  Label Co-occurrence Heatmap ────────────────────────────────────────
cooc = y_food.T @ y_food
fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(
    cooc, xticklabels=FOOD_CATEGORIES, yticklabels=FOOD_CATEGORIES,
    cmap='YlOrRd', annot=True, fmt='d', linewidths=0.4, ax=ax
)
ax.set_title('Food Behavior Label Co-occurrence Matrix')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('/content/food_label_cooccurrence.png', dpi=120)
plt.show()

---
## Step 7 — Save All Artifacts to Google Drive

In [ ]:
import shutil

# Save model and supporting objects
artifacts = {
    'food_behavior_model.pkl': food_model,
    'food_mlb.pkl':            food_mlb,
}
for fname, obj in artifacts.items():
    path = os.path.join(SAVE_DIR, fname)
    with open(path, 'wb') as f:
        pickle.dump(obj, f)
    print(f'Saved: {path}')

# Save the lookup table and category list
shutil.copy('/content/food_lookup.json', os.path.join(SAVE_DIR, 'food_lookup.json'))

with open(os.path.join(SAVE_DIR, 'food_categories.json'), 'w') as f:
    json.dump(FOOD_CATEGORIES, f)

# Save plots
for png in ['food_label_distribution.png', 'food_metrics_bar.png',
            'food_per_class_f1.png', 'food_label_cooccurrence.png']:
    src = f'/content/{png}'
    if os.path.exists(src):
        shutil.copy(src, os.path.join(SAVE_DIR, png))

# Save BERT model name so notebook 3 can reload it
with open(os.path.join(SAVE_DIR, 'bert_model_name.txt'), 'w') as f:
    f.write(BERT_MODEL)

print('\nAll food-model artifacts saved to Google Drive.')
print('Contents:', os.listdir(SAVE_DIR))